# anticipated群：ARIS care 最初の該当記載日の再抽出
- 対象: unex_study.unex_death = 0
- 検索範囲: karte 全期間〜死亡日（死亡日当日を含む）
- Phase 1: 全ヒット抽出 → 目視確認用CSV
- Phase 2: 目視確認後 → first_date 確定、24h/48h/72h/7日フラグ

In [9]:
# 0. 設定
import sqlite3, re, unicodedata
import numpy as np
import pandas as pd
from pathlib import Path

DB  = "/Users/muna/Hana_research/data/db/Hana_Research.db"
OUT = Path("/Users/muna/Hana_research/data/output/aris_firstdate")
OUT.mkdir(parents=True, exist_ok=True)

# 死亡確認カルテの判定（Cell 2 の visit_type 一覧を見て修正）
DEATH_VISIT_TYPES = ["緊急", "往診", "休日", "定期", "深夜", "夜間往診"]   # 部分一致
DEATH_WORDS       = ["死亡確認", "死亡診断", "死亡を確認", "永眠"]        # 要確認・追加可
NURSE_TYPE        = "訪問看護"

THRESH_H  = {"24h": 24, "48h": 48, "72h": 72, "7d": 168}
YOTEI_WIN = 15   # ARIS表記の前後何文字以内の「予定」を yotei=1 とするか

# NFKC正規化後に適用（全角ＡＲＩＳ・半角ｱﾘｽ も吸収）
ARIS_RE = re.compile(r"(?<![A-Za-z])A?RIS(?![A-Za-z])|アリス", re.I)

def nfkc(x):
    return unicodedata.normalize("NFKC", x) if isinstance(x, str) else ""

In [10]:
# 1. 対象患者と死亡日
con = sqlite3.connect(DB)

coh = pd.read_sql("SELECT Patient_ID, Study_ID, unex_death FROM unex_study", con)
print("unex_study:", len(coh))
print(coh["unex_death"].value_counts(dropna=False))

ev = pd.read_sql("""
    SELECT DISTINCT Study_ID, death_date FROM event
    WHERE death_date IS NOT NULL AND death_date <> ''
""", con)
ev["death_date"] = pd.to_datetime(ev["death_date"], errors="coerce").dt.normalize()
ev = ev.dropna(subset=["death_date"]).drop_duplicates()
dup = ev[ev["Study_ID"].duplicated(keep=False)]
assert dup.empty, f"1患者に複数のdeath_date:\n{dup}"

ant = coh[coh["unex_death"] == 0].merge(ev, on="Study_ID", how="left")
print("anticipated:", len(ant), "/ death_date欠損:", ant["death_date"].isna().sum())

con.execute("DROP TABLE IF EXISTS temp.ids")
con.execute("CREATE TEMP TABLE ids (Study_ID TEXT PRIMARY KEY)")
con.executemany("INSERT INTO temp.ids VALUES (?)", [(s,) for s in ant["Study_ID"]])

unex_study: 1281
unex_death
0    1042
1     239
Name: count, dtype: int64
anticipated: 1042 / death_date欠損: 0


In [11]:
# 2. visit_type の確認（DEATH_VISIT_TYPES の修正用）
print(pd.read_sql("""
    SELECT k.visit_type, COUNT(*) AS n
    FROM karte k JOIN temp.ids i ON k.Study_ID = i.Study_ID
    GROUP BY k.visit_type ORDER BY n DESC
""", con).to_string())

   visit_type      n
0        定期訪問  42132
1        電話再診  12940
2         その他  11979
3          文書   6904
4        訪問看護   4214
5        緊急往診   3053
6    訪問看護(介護)   2100
7        予防接種   2056
8          自費   1284
9          外来   1139
10         点滴   1039
11   訪問リハ(介護)    980
12         初診    953
13         往診    871
14       休日往診    845
15       夜間往診    751
16       分割記載    589
17       深夜往診    483
18         なし    456
19     再診(0点)    330
20         報告    287
21       訪問リハ    184
22  再診(オンライン)    173
23         会議    116
24      退院時会議     82
25         検査     43
26       外来リハ     21
27       訪問栄養     18
28   訪問栄養(介護)     16
29         再診      9
30      コンサルト      3
31       栄養指導      1
32  初診(オンライン)      1


In [12]:
# 3. カルテ読込・日時パース・死亡日までに限定
kt = pd.read_sql("""
    SELECT k.record_no, k.Study_ID, k.visit_type, k.visit_datetime, k.karte_text
    FROM karte k JOIN temp.ids i ON k.Study_ID = i.Study_ID
""", con)

s = (kt["visit_datetime"].astype(str)
       .str.replace(r"\([^)]*\)", " ", regex=True)
       .str.replace(r"\s+", " ", regex=True).str.strip())
kt["rec_dt"]   = pd.to_datetime(s, format="%Y/%m/%d %H:%M", errors="coerce")
kt["rec_date"] = pd.to_datetime(s.str.extract(r"^(\d{4}/\d{1,2}/\d{1,2})")[0],
                                format="%Y/%m/%d", errors="coerce")
print("日付パース失敗:", kt["rec_date"].isna().sum(), "/ 時刻パース失敗:", kt["rec_dt"].isna().sum())
print(kt.loc[kt["rec_date"].isna(), "visit_datetime"].head(10).tolist())

kt = kt.merge(ant[["Study_ID", "death_date"]], on="Study_ID", how="inner")
kt = kt[kt["rec_date"] <= kt["death_date"]].copy()
kt["source"] = np.where(kt["visit_type"] == NURSE_TYPE, "看護", "医師")
kt["text_n"] = kt["karte_text"].map(nfkc)
print("対象カルテ:", len(kt), "件 /", kt["Study_ID"].nunique(), "例")

日付パース失敗: 0 / 時刻パース失敗: 2
[]
対象カルテ: 95642 件 / 1037 例


In [13]:
# 4. 死亡時刻（当院の死亡確認カルテの診察時刻）
vt_pat = "|".join(map(re.escape, DEATH_VISIT_TYPES))
dw_pat = "|".join(map(re.escape, DEATH_WORDS))
dd = kt[(kt["rec_date"] == kt["death_date"])
        & (kt["source"] == "医師")
        & kt["visit_type"].fillna("").str.contains(vt_pat)
        & kt["text_n"].str.contains(dw_pat)]
death_time = dd.groupby("Study_ID")["rec_dt"].min().rename("death_dt")
ant = ant.drop(columns="death_dt", errors="ignore").merge(death_time, on="Study_ID", how="left")
print("死亡時刻あり:", ant["death_dt"].notna().sum(), "/ なし（日単位判定）:", ant["death_dt"].isna().sum())

死亡時刻あり: 593 / なし（日単位判定）: 449


In [14]:
# 5. ARISヒット抽出（1行=1カルテ記録）
rows = []
for r in kt.itertuples(index=False):
    ms = list(ARIS_RE.finditer(r.text_n))
    if not ms:
        continue
    t = r.text_n
    yotei = any("予定" in t[max(0, m.start() - YOTEI_WIN): m.end() + YOTEI_WIN] for m in ms)
    m0 = ms[0]
    rows.append(dict(
        Study_ID=r.Study_ID, record_no=r.record_no, source=r.source, visit_type=r.visit_type,
        rec_dt=r.rec_dt, rec_date=r.rec_date, n_match=len(ms),
        matches="|".join(sorted({m.group() for m in ms})), yotei=int(yotei),
        context=t[max(0, m0.start() - 60): m0.end() + 60].replace("\n", " "),
    ))
hits = pd.DataFrame(rows)
hits = hits.merge(ant[["Study_ID", "death_date", "death_dt"]], on="Study_ID", how="left")
hits["lead_days"]  = (hits["death_date"] - hits["rec_date"]).dt.days
hits["lead_hours"] = ((hits["death_dt"] - hits["rec_dt"]).dt.total_seconds() / 3600).round(1)
hits = hits.sort_values(["Study_ID", "rec_dt", "record_no"]).reset_index(drop=True)
hits["hit_order"] = hits.groupby("Study_ID").cumcount() + 1
print("ヒット記録:", len(hits), "/ ヒットあり患者:", hits["Study_ID"].nunique(), "/", len(ant))
print(hits["matches"].value_counts().head(20))

ヒット記録: 2362 / ヒットあり患者: 387 / 1042
matches
ARIS         2273
ARIS|RIS       44
アリス            41
ARIS|ARIs       2
Aris            1
RIS             1
Name: count, dtype: int64


In [15]:
# 6. 目視確認用CSV出力
review = hits.copy()
review.insert(0, "valid", "")   # 1=該当, 0=偽陽性。患者ごとに hit_order=1 から確認し、最初の1が出たら以降は空欄でよい
review.to_csv(OUT / "aris_hits_review.csv", index=False, encoding="utf-8-sig")

nohit = ant[~ant["Study_ID"].isin(hits["Study_ID"])]
nohit.to_csv(OUT / "aris_nohit.csv", index=False, encoding="utf-8-sig")
print("ヒットなし（据え置き）:", len(nohit))

ヒットなし（据え置き）: 655


---
## Phase 2：aris_hits_review.csv の valid 列を記入後に実行

In [16]:
# 7. first_date確定と閾値フラグ
rev = pd.read_csv(OUT / "aris_hits_review.csv", encoding="utf-8-sig",
                  parse_dates=["rec_dt", "rec_date", "death_date", "death_dt"])
rev["valid"] = pd.to_numeric(rev["valid"], errors="coerce")

first = (rev[rev["valid"] == 1].sort_values(["Study_ID", "rec_dt"])
            .groupby("Study_ID").head(1)
            .rename(columns={"rec_dt": "first_dt", "rec_date": "first_date"}))

# 確定した first より前に未確認（空欄）の記録が残っていないか
chk = rev.merge(first[["Study_ID", "hit_order"]].rename(columns={"hit_order": "first_order"}), on="Study_ID")
unchecked = chk[(chk["hit_order"] < chk["first_order"]) & chk["valid"].isna()]
print("first より前の未確認記録:", len(unchecked))

# 全ヒットが0（偽陽性のみ）の患者
all_false = set(rev["Study_ID"]) - set(first["Study_ID"]) - set(rev.loc[rev["valid"].isna(), "Study_ID"])
print("全ヒット偽陽性:", len(all_false))

res = ant.merge(first[["Study_ID", "first_dt", "first_date", "source", "yotei",
                       "lead_days", "lead_hours", "context"]], on="Study_ID", how="left")
has_time = res["death_dt"].notna() & res["first_dt"].notna()
res["judge_unit"] = np.where(res["first_date"].isna(), "", np.where(has_time, "hour", "day"))

for k, h in THRESH_H.items():
    d = h // 24
    ge = np.where(has_time, res["lead_hours"] >= h, res["lead_days"] >= d).astype(float)
    ge[res["first_date"].isna().to_numpy()] = np.nan
    res[f"ge_{k}"] = ge
    res[f"border_{k}"] = ((~has_time) & (res["lead_days"] == d)).astype(int)

res.to_csv(OUT / "aris_firstdate_result.csv", index=False, encoding="utf-8-sig")

print(res["lead_days"].describe())
summ = pd.DataFrame({k: [int(np.nansum(res[f"ge_{k}"])), int(res[f"ge_{k}"].notna().sum()),
                         int(res[f"border_{k}"].sum())] for k in THRESH_H},
                    index=["記載あり(≥閾値)", "first_dateあり", "境界フラグ"])
print(summ)

first より前の未確認記録: 0
全ヒット偽陽性: 0
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: lead_days, dtype: float64
              24h  48h  72h  7d
記載あり(≥閾値)       0    0    0   0
first_dateあり    0    0    0   0
境界フラグ           0    0    0   0
